In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

from matplotlib.gridspec import GridSpec
#from ipywidgets import interact, FloatSlider, FloatLogSlider
import warnings
warnings.filterwarnings('ignore')
import os

from scipy.ndimage import gaussian_filter

import visualize_local as vis
import matplotlib.colors as mcolors

# visualize paraview linecuts

In [ ]:
def get_slice(sample, coord, date, data_dir):
    """
    Returns a dict of all parameters/paths needed to load a single
    (sample, slice) combination. 
    """

    cfg = dict(
        sample=sample,
        coord = coord
    )

    print(cfg)

    date = date 

    if sample == '665':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 51 #nm
    elif sample == '270':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 66 #nm
    elif sample == '452':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 33 #nm
    elif sample == '105':
        cfg['center1'] = 257
        cfg['film_thickness_nm'] = 50 #nm ################################ THIS IS WRONG NEED TO UPDATE 

    else:
        raise ValueError(f"Unrecognized sample '{sample}'")

    file_path = os.path.join(data_dir, f"{sample}_{coord}_center_slice_{date}.csv")
    data = np.genfromtxt(file_path, delimiter=",", names=True)

    if coord == 'y':
        coord1  = data['Points0']
        coord2  = data['Points2']
        mz = data['magnetization2'] 
        mx = data['magnetization1'] ############# I changed this based on observation, may be wrong
        my = data['magnetization0']

    elif coord == 'x':
        coord1  = data['Points1']
        coord2  = data['Points2'] 
        mz = data['magnetization2'] 
        mx = data['magnetization0'] ############# I changed this based on observation, may be wrong
        my = data['magnetization1']
    
    eps = 1e-12
    mnorm = np.sqrt(mx**2 + my**2 + mz**2) + eps
    mxn = mx / mnorm
    myn = my / mnorm
    mzn = mz / mnorm
    
    theta = np.arccos(np.clip(mzn, -1.0, 1.0))   

    cfg['slice'] = data
    cfg['x'] = coord1
    cfg['z'] = coord2
    cfg['mx'] = mx
    cfg['my'] = my
    cfg['mz'] = mz
    cfg['mxn'] = mxn
    cfg['myn'] = myn
    cfg['mzn'] = mzn
    cfg['theta'] = theta
    cfg['mnorm'] = mnorm


    return cfg

def straighten_and_store(cfg, field_pairs, n_show):
    for field, field_n in field_pairs:
        unique_x, theta_grid, grid, grid_n, grid_normall, n_cols, eta_targets = vis.straighten_data(
            cfg['x'], cfg['z'], cfg['theta'],
            cfg[field], cfg[field_n], cfg['mnorm'], n_show=n_show
        )

        cfg.setdefault('unique_x', unique_x)
        cfg.setdefault('theta_grid', theta_grid)
        cfg.setdefault('n_cols', n_cols)
        cfg.setdefault('eta_targets', eta_targets)
        cfg.setdefault('M_mag', grid_normall)
        

        cfg[f'{field}_grid'] = grid
        cfg[f'{field_n}_grid'] = grid_n

    return cfg

In [ ]:
### User set up 

# -----------------------------------------------------------------------
# Scan To Analyze
# -----------------------------------------------------------------------

sample_ids = ['665', '270', '452', '105']
dates = ['11JULY26', '11JULY26', '11JULY26', '11JULY26' ]
#base_path = '/homes/emma/notebooks/RawData/{sample}/vectors/'
#base_path = r'C:\Users\emmab\Projects\Vector_Tomography\665_z_center_slice_1JULY26.csv' 
base_path = ''

how_many_slices = 5 # (steps in percentage)
field_pairs = [('mz', 'mzn'), ('my', 'myn'), ('mx', 'mxn')]



samples = {}
for sample, date in zip(sample_ids, dates):
    samples[sample] = {
        'y': get_slice(sample, 'y', date, base_path.format(sample=sample)),
        'x': get_slice(sample, 'x', date, base_path.format(sample=sample)),
        # 'z': get_slice(sample, 'z', date, base_path.format(sample=sample)),
    }

for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        coords[coord] = straighten_and_store(data_dict, field_pairs, n_show=how_many_slices)

# -----------------------------------------------------------------------
# Plot set up
# -----------------------------------------------------------------------
colors = plt.cm.viridis(np.linspace(0, 1, how_many_slices))

plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'lines.linewidth': 2.2,
    'axes.grid': False,
    'grid.alpha': 0.3,
})


# full profile

In [ ]:
# print(samples['270']['y'].keys())
view = samples['270']['x']
#plot_full_plot(view,field_pairs,colors)


#print(samples['270']['x'].keys())
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        vis.plot_magnitude(samples[sample][coord], colors)
        

# select one wall

In [ ]:
crop1_ranges = {
    '665': {'x': (495, 508), 'y': (485, 515)},
    '270': {'x': (590, 615), 'y': (575, 590)},
    '452': {'x': (515, 525), 'y': (415, 435)},
    '105': {'x': (503, 520), 'y': (540, 555)},
}

crop1 = {}
for sample, coords in samples.items():
    crop1[sample] = {}
    for coord, data_dict in coords.items():
        xmin, xmax = crop1_ranges[sample][coord]
        crop1[sample][coord] = {
            'dw': vis.slice_straighten_and_store(data_dict, (xmin, xmax), field_pairs, n_show=how_many_slices),
            'xmin': xmin,
            'xmax': xmax,
        }



In [ ]:
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        if sample == '665':
            plot_grad = True
        else: 
            plot_grad = False

        crop1[sample][coord]['dw'] = vis.gradient(crop1[sample][coord]['dw'], field_pairs, colors, smooth=True, normalized = False, plot=plot_grad)

# simulate walls

In [ ]:
### --- Define wall types and their parameters --- ###
delta_demo = 2.5

xmin, xmax = 485, 515
#xmax, xmin = crop1_list[0][1], crop1_list[0][0]
x0 = np.linspace(0, (xmax - xmin), (xmax - xmin) + 1)
x0 = x0 - x0.mean()  # recenter around 0


# wall_types = {
#     'bloch':  {'psi': np.pi / 2, 'color': 'magenta', 'linstyle' : '--'},
#     'neel':   {'psi': 0,          'color': 'cyan', 'linstyle' : '-.'},
#     'hybrid': {'psi': np.pi / 4,  'color': 'darkviolet', 'linstyle' : ':'}} 
# ### --- Compute and store Mx, My, Mz for each wall type ---###
# for wall, cfg in wall_types.items():
#     print(wall)
#     print(cfg['psi'])
#     Mx, My, Mz = vis.magnetization_components(x0, delta_demo, cfg['psi'])
#     cfg['Mx'] = Mx
#     cfg['My'] = My
#     cfg['Mz'] = Mz

#     dx0       = x0[1] - x0[0]
#     dMz_dx   = np.gradient(Mz, dx0)
#     dMy_dx   = np.gradient(My, dx0)
#     dMx_dx   = np.gradient(Mx, dx0)
#     cfg['dMx_dx'] = dMx_dx
#     cfg['dMy_dx'] = dMy_dx
#     cfg['dMz_dx'] = dMz_dx

wall_types = {
    'bloch':  {'psi': np.pi / 2, 'color': 'magenta'},
    'neel':   {'psi': 0,          'color': 'cyan'},
    'hybrid': {'psi': np.linspace(-np.pi /1.05, np.pi /1.1, 11), 'cmap': plt.cm.twilight_shifted},
}


for wall, cfg in wall_types.items():
    psi = cfg['psi']
    if np.isscalar(psi):
        Mx, My, Mz = vis.magnetization_components(x0, delta_demo, psi)
        cfg['Mx'], cfg['My'], cfg['Mz'] = Mx, My, Mz

        dx0       = x0[1] - x0[0]
        dMz_dx   = np.gradient(Mz, dx0)
        dMy_dx   = np.gradient(My, dx0)
        dMx_dx   = np.gradient(Mx, dx0)
        cfg['dMx_dx'] = dMx_dx
        cfg['dMy_dx'] = dMy_dx
        cfg['dMz_dx'] = dMz_dx
    
    else:
        # hybrid: sweep over multiple psi values, store one array per psi
        Mx_list, My_list, Mz_list = [], [], []
        dMx_dx_list, dMy_dx_list, dMz_dx_list = [], [], []
        dx0       = x0[1] - x0[0]
        for psi_val in psi:
            Mx, My, Mz = vis.magnetization_components(x0, delta_demo, psi_val)
            Mx_list.append(Mx)  # matches your sign flip in the sweep
            My_list.append(My)
            Mz_list.append(Mz)
            dMz_dx_list.append(np.gradient(Mz, dx0))
            dMy_dx_list.append(np.gradient(My, dx0))
            dMx_dx_list.append(np.gradient(Mx, dx0))

        cfg['Mx'], cfg['My'], cfg['Mz'] = Mx_list, My_list, Mz_list
        cfg['dMx_dx'], cfg['dMy_dx'], cfg['dMz_dx'] = dMx_dx_list, dMy_dx_list, dMz_dx_list
        cfg['colors'] = cfg['cmap'](np.linspace(0, 1, len(psi)))

In [ ]:
   
for wall, cfg in wall_types.items():
    vis.plot_wall_components(wall, cfg, x0)
    vis.plot_wall_gradients(wall, cfg, x0)


# domain wall width

In [ ]:
# Functions 
def fit_linecut_hyperbolic(x, Mz, z_shift, xmax, xmin, sample, plot = True): 
    # range_px = xmax - xmin
    # pixel_size_nm   = 8         # voxel size in nm  
    # # range_nm = range_px*pixel_size_nm
    # # print(range_nm)
    # # xnm = np.arange(len(range_nm)) 
    # x_nm = x * pixel_size_nm
    
    pixel_size_nm = 8  # voxel size in nm
    x_data = x - x.mean()
    x_nm = x_data*pixel_size_nm

    # range_px = xmax - xmin
    # range_nm = range_px * pixel_size_nm

    # # Center x (in pixels) around 0, then convert to nm
    # center_px = xmin + range_px / 2
    # x_nm = (x - center_px) * pixel_size_nm

    colors = plt.cm.viridis(np.linspace(0, 1, len(z_shift)))
    
    dw_width = []
    dw_center = []

    fig = plt.figure(figsize=(21, 6))
    fig.suptitle(f'Hyperbolic Tan Fit of Data for range ({xmin}, {xmax}) \n {sample}', fontsize=20, y=1.0)
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.set_xlabel('cross section (nm)')
    ax1.set_ylabel(r'$M_z / M_{norm}$')
    ax1.set_title('Line cuts of domains at different z')
    
    ax2 = fig.add_subplot(1,3, 2)
    ax2.set_xlabel('slice (%)')
    ax2.set_ylabel('DW width')
    ax2.set_title('fit DW width at different slices')


    ax3 = fig.add_subplot(1, 3, 3)
    ax3.set_xlabel('slice (%)')
    ax3.set_ylabel('DW center')
    ax3.set_title('fit DW center at different slices')


    for j, eta_v in enumerate(z_shift):

        # Linecut
        ax1.plot(x_nm, Mz[j, :], color=colors[j], lw=1.5, marker='o', label=fr'{eta_v*100:.0f}%')
         
        # Fit with hyperbolic tangent
        hyperbolic_fit, hyperbolic_error = vis.fit_hyperbolic(x_nm, Mz[j, :])
        hyperbolic_plot = vis.hyperbolic(x_nm, *hyperbolic_fit)
        dw_width.append(hyperbolic_fit[2]) ## might need to be converted into nm
        dw_center.append(hyperbolic_fit[1])
        
        # Overlay linecut with fit 
        ax1.plot(x_nm, hyperbolic_plot, color=colors[j], linewidth=3, linestyle = '--') #, label=f'Hyperbolic Fit \n DW width = {hyperbolic_fit[2]:.2f} nm')

    ax1.legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc = 'upper right')
    ax2.plot(z_shift*100, np.abs(dw_width), marker='o')
    ax2.axhline(np.mean(np.abs(dw_width)), color = 'red', label = f'mean Δ = {np.mean(dw_width):.2f}')
    ax2.legend()
    ax3.plot(z_shift*100, dw_center, marker='o')
    
    plt.tight_layout()
    plt.show()

    return np.abs(dw_width), dw_center

In [ ]:
dw_fits = {}
        
for sample, coords in samples.items():
    dw_fits[sample] = {}
    for coord, data_dict in coords.items():

        wall = crop1[sample][coord]['dw']
        dw_width, dw_center = fit_linecut_hyperbolic(
            wall['unique_x'],
            wall['mzn_grid'],
            wall['eta_targets'],
            crop1[sample][coord]['xmax'],
            crop1[sample][coord]['xmin'],
            sample=sample,
            plot=True
        )

        dw_fits[sample][coord] = {
            'dw_width': dw_width,
            'dw_center': dw_center,
            'eta_targets': wall['eta_targets'],
        }

print(dw_fits['665']['x'].keys())
print(dw_fits['665']['y'].keys())

# print(dw_fits['665']['dw_width'])
        
# print(np.array(dw_width)/8)
# print(np.array(dw_center)/8)

In [ ]:
## Compare x and y slice 
sample_colors = ['gold', 'deeppink', 'limegreen', 'cyan']
# for sample, coords in samples.items():
#     fig, ax = plt.subplots(figsize=(10, 6))
#     for coord in coords:
#         fit = dw_fits[sample][coord]
#         ax.plot(fit['eta_targets'] * 100, fit['dw_width'], marker='o', label=f'{coord}-slice')

#     ax.set_xlabel('slice (%)')
#     ax.set_ylabel('DW width (nm)')
#     ax.set_title(f'Fit DW width for {sample}, comparing x vs y slices (Mz)')
#     ax.legend(loc='upper right')
#     plt.tight_layout()
#     plt.show()

## Compare sample per x and y  

for coord in ['x', 'y']:
    fig, ax = plt.subplots(figsize=(10, 6))
    for sample, color in zip(samples, sample_colors):
        fit = dw_fits[sample][coord]
        ax.plot(fit['eta_targets'] * 100, fit['dw_width'], color = color, marker='o', label=f'{sample}')

    ax.set_xlabel('slice (%)')
    ax.set_ylabel('DW width (nm)')
    ax.set_title(f'Fit DW width across samples, {coord}-slice (Mz)')
    ax.legend(loc='upper right', title='sample')
    plt.tight_layout()
    plt.show()

fit theta per x in thickness

In [ ]:
### we know that theta(x) = arccos(mz)
def plot_theta(cfg, colors):
    fig, ax = plt.subplots(figsize=(7, 7))
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    for j, eta_v in enumerate(cfg['eta_targets']):
        ax.plot(x_data, cfg['theta_grid'][j,:]/np.pi, color=colors[j], marker = 'o', lw=1.5, label=fr'{eta_v*100:.0f}%')
    
    ax.set_ylabel(r'θ(x) = arccos($m_z$) (1/π)')
    ax.set_xlabel('x (pix)')
    ax.axhline(1, color = 'crimson', label = 'sin(θ) = 0!!')

    ax.legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc = 'upper left')
    ax.set_title(f"Calculated theta per x \n {cfg['sample']} {cfg['coord']}-slice")



for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        plot_theta(crop1[sample][coord]['dw'], colors)



# calculate psi

In [ ]:
def plot_psi(cfg, colors, n_bins = 11):
    fig, ax = plt.subplots(1,3, figsize=(25, 8))
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()

    # same binning setup as plot_by_psi_bin, so colors/edges match exactly
    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    for j, eta_v in enumerate(cfg['eta_targets']):
        ax[0].plot(x_data, cfg['mx_psi_grid'][j, :] / np.pi, color=colors[j], lw=1.5,
                marker='o', label=fr'ψ from $M_x$, {eta_v*100:.0f}%')
        ax[1].plot(x_data, cfg['my_psi_grid'][j, :] / np.pi, color=colors[j], lw=1.5,
                marker='s', label=fr'ψ from $M_y$, {eta_v*100:.0f}%')
        # #ax[0].plot(x_data, cfg['psi_grid'][j, :] / np.pi, color=colors[j], lw=2.5,
        #         linestyle=':', label=fr'ψ (atan2), {eta_v*100:.0f}%')
        # #ax[1].plot(x_data, cfg['psi_grid'][j, :] / np.pi, color=colors[j], lw=2.5,
        #         linestyle=':', label=fr'ψ (atan2), {eta_v*100:.0f}%')
        ax[2].plot(x_data, cfg['psi_grid'][j, :] / np.pi, color=colors[j], lw=2.5,
                marker='o', linestyle=':', label=fr'ψ (atan2), {eta_v*100:.0f}%')

    for a in ax:
        for k in range(n_bins):
            a.axhspan(bin_edges[k], bin_edges[k+1], color=bin_colors[k], alpha=0.15, zorder=0)

    ax[0].set_ylabel(r'ψx(x) (1/π)')
    ax[1].set_ylabel(r'ψy(x) (1/π)')
    ax[0].set_xlabel('x (pix)')
    ax[0].set_ylim(-1,1)
    ax[0].axhline(0, color='cyan', label='Néel wall')
    ax[0].axhline(0.5, color='magenta', label='Bloch wall')
    ax[0].axhline(-0.5, color='magenta')
    ax[0].legend(fontsize=7, ncol=2, title='fraction of local thickness', loc='lower left')
    ax[0].set_title(f"Calculated ψ per x \n {cfg['sample']} {cfg['coord']}-slice")
        
    ax[1].set_xlabel('x (pix)')
    ax[1].set_ylim(-1,1)
    ax[1].axhline(0, color='cyan', label='Néel wall')
    ax[1].axhline(0.5, color='magenta', label='Bloch wall')
    ax[1].axhline(-0.5, color='magenta')
    ax[1].legend(fontsize=7, ncol=2, title='fraction of local thickness', loc='lower left')
    ax[1].set_title(f"Calculated ψ per x \n {cfg['sample']} {cfg['coord']}-slice")

    ax[2].set_ylabel(r'ψ(x) (1/π)')
    ax[2].set_xlabel('x (pix)')
    ax[2].set_ylim(-1,1)
    ax[2].axhline(0, color='cyan', label='Néel wall')
    ax[2].axhline(0.5, color='magenta', label='Bloch wall')
    ax[2].axhline(-0.5, color='magenta')
    ax[2].legend(fontsize=7, ncol=2, title='fraction of local thickness', loc='lower left')
    ax[2].set_title(f"Calculated ψ per x \n {cfg['sample']}")

    plt.tight_layout()
    plt.show()

def calc_psi(cfg, normalized = True, plot=True):
    sin_theta = np.sin(cfg['theta_grid'])

    if np.any(np.isclose(sin_theta, 0)):
        print('Warning: sin(theta) near zero at some points — psi undefined there')

    if normalized:
        mx_component = cfg['mxn_grid'] / sin_theta
        my_component = cfg['myn_grid'] / sin_theta
    else:
        mx_component = cfg['mx_grid'] / sin_theta
        my_component = cfg['my_grid'] / sin_theta

    # clip to [-1, 1] to guard against floating-point drift just outside that range,
    # which would otherwise produce NaN from arccos/arcsin
    mx_component_clipped = np.clip(mx_component, -1, 1)
    my_component_clipped = np.clip(my_component, -1, 1)

    cfg['mx_psi_grid'] = np.arccos(mx_component_clipped)
    cfg['my_psi_grid'] = np.arcsin(my_component_clipped)
    cfg['psi_grid'] = np.arctan2(my_component, mx_component)  # quadrant-correct reference

    if plot:
        plot_psi(cfg, colors)

    return cfg



for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        calc_psi(crop1[sample][coord]['dw'], normalized = False)

### plot for full stack 
for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        calc_psi(samples[sample][coord], plot = False)


### create 5 separate quadrants per psi (evenly spaced between -1 and 1) and associate x values that fall into that quadrent with a color. Per x and its associated color, I want to replot mz, mx, and my as a function of x. 

In [ ]:
def plot_by_psi_bin(cfg, select_psi = None, n_bins=11):
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_over_pi = cfg[select_psi+'psi_grid'] / np.pi  # units of ψ/π, CHANGED TO MY PSI

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    #markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']  # cycles if you have more slices than markers

    fig, ax = plt.subplots(1, 3, figsize=(21, 6))

    for j, eta_v in enumerate(cfg['eta_targets']):
        psi_slice = psi_over_pi[j, :]


        ax[0].plot(x_data, cfg['mzn_grid'][j, :], c=colors[j], linewidth = 1, label=fr'{eta_v*100:.0f}%')
        ax[1].plot(x_data, cfg['myn_grid'][j, :], c=colors[j], linewidth = 1, label=fr'{eta_v*100:.0f}%')
        ax[2].plot(x_data, cfg['mxn_grid'][j, :], c =colors[j], linewidth = 1, label=fr'{eta_v*100:.0f}%')
        


        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx]

        # marker = markers[j % len(markers)]

        ax[0].scatter(x_data, cfg['mzn_grid'][j, :], c=point_colors) #, marker=marker, s=25)
        ax[1].scatter(x_data, cfg['myn_grid'][j, :], c=point_colors) #, marker=marker, s=25)
        ax[2].scatter(x_data, cfg['mxn_grid'][j, :], c=point_colors) #, marker=marker, s=25)

    ax[0].set_ylabel(r'$M_{z}$')
    ax[1].set_ylabel(r'$M_{y}$')
    ax[2].set_ylabel(r'$M_{x}$')
    for a in ax:
        a.set_xlabel('x (pix)')

    fig.suptitle(f"{cfg['sample']} {cfg['coord']}-slice — colored by {select_psi} ψ")

    # ψ-bin color legend
    color_handles = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=bin_colors[k], markersize=8,
                   label=fr'ψ∈[{bin_edges[k]:.1f}, {bin_edges[k+1]:.1f}]π')
        for k in range(n_bins)
    ]
    ## slice marker-shape legend
    # marker_handles = [
    #     plt.Line2D([0], [0], marker=markers[j % len(markers)], color='k', linestyle='None',
    #                markersize=4, label=fr'{eta_v*100:.0f}%')
    #     for j, eta_v in enumerate(cfg['eta_targets'])
    # ]

    # ax[2].legend(handles=color_handles, fontsize=6, ncol=2,
    #              loc='upper right', title='ψ bin')

    ax[0].legend(fontsize=6, ncol=2, loc='upper right', title='Fraction of local thickness')
    ax[1].legend(fontsize=6, ncol=2, loc='upper right', title='Fraction of local thickness')
    ax[2].legend(fontsize=6, ncol=2, loc='upper right', title='Fraction of local thickness')

    plt.tight_layout()
    plt.show()


# for sample, coords in crop1.items():
#     for coord, crop_info in coords.items():
#         plot_by_psi_bin(crop_info['dw'], select_psi = '' )

In [ ]:
def plot_gradient_by_psi_bin(cfg, select_psi = None, normalized = True, n_bins=11):
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_over_pi = cfg[select_psi+'psi_grid'] / np.pi  # units of ψ/π, matching your existing convention

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(1, 3, figsize=(21, 6))


    if normalized: 
        x = 'dmxn_dx'
        y = 'dmyn_dx'
        z = 'dmzn_dx'
    else: 
        x = 'dmx_dx'
        y = 'dmy_dx'
        z = 'dmz_dx'


    for j, eta_v in enumerate(cfg['eta_targets']):
        psi_slice = psi_over_pi[j, :]


        ax[0].plot(x_data, cfg[z][j, :], c=colors[j], linewidth = 1, label=fr'{eta_v*100:.0f}%')
        ax[1].plot(x_data, cfg[y][j, :], c=colors[j], linewidth = 1, label=fr'{eta_v*100:.0f}%')
        ax[2].plot(x_data, cfg[x][j, :], c =colors[j], linewidth = 1, label=fr'{eta_v*100:.0f}%')
        


        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx]

        # marker = markers[j % len(markers)]

        ax[0].scatter(x_data, cfg[z][j, :], c=point_colors) #, marker=marker, s=25)
        ax[1].scatter(x_data, cfg[y][j, :], c=point_colors) #, marker=marker, s=25)
        ax[2].scatter(x_data, cfg[x][j, :], c=point_colors) #, marker=marker, s=25)

    ax[0].set_ylabel(r'd$M_{z}$/dx')
    ax[1].set_ylabel(r'd$M_{y}$/dx')
    ax[2].set_ylabel(r'd$M_{x}$/dx')
    for a in ax:
        a.set_xlabel('x (pix)')

    fig.suptitle(f"Gradient of {cfg['sample']} {cfg['coord']}-slice — colored by {select_psi} ψ")

    # ψ-bin color legend
    color_handles = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=bin_colors[k], markersize=8,
                   label=fr'ψ∈[{bin_edges[k]:.1f}, {bin_edges[k+1]:.1f}]π')
        for k in range(n_bins)
    ]
    ## slice marker-shape legend
    # marker_handles = [
    #     plt.Line2D([0], [0], marker=markers[j % len(markers)], color='k', linestyle='None',
    #                markersize=4, label=fr'{eta_v*100:.0f}%')
    #     for j, eta_v in enumerate(cfg['eta_targets'])
    # ]

    # ax[2].legend(handles=color_handles, fontsize=6, ncol=2,
    #              loc='upper right', title='ψ bin')

    ax[0].legend(fontsize=6, ncol=2, loc='lower left', title='Fraction of local thickness')
    ax[1].legend(fontsize=6, ncol=2, loc='lower left', title='Fraction of local thickness')
    ax[2].legend(fontsize=6, ncol=2, loc='lower left', title='Fraction of local thickness')

    plt.tight_layout()
    plt.show()


for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        plot_gradient_by_psi_bin(crop_info['dw'], select_psi = 'my_', normalized = False)
        plot_gradient_by_psi_bin(crop_info['dw'], select_psi = 'mx_', normalized = False)
        plot_gradient_by_psi_bin(crop_info['dw'], select_psi = '', normalized = False)

In [ ]:
def plot_by_psi_bin_offset(cfg, select_psi = None, normalized = True,  n_bins=11, offset=1.5):
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_over_pi = cfg[select_psi+'psi_grid'] / np.pi  # units of ψ/π, matching your existing convention

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(1, 3, figsize=(21, 8))

    n_slices = len(cfg['eta_targets'])


    if normalized: 
        x = 'mxn_grid'
        y = 'myn_grid'
        z = 'mzn_grid'
    else: 
        x = 'mx_grid'
        y = 'my_grid'
        z = 'mz_grid'
    

    for j, eta_v in enumerate(cfg['eta_targets']):
        psi_slice = psi_over_pi[j, :]

        # shift this slice's baseline up by j * offset
        shift = j * offset

        mzn = cfg[z][j, :] + shift
        myn = cfg[y][j, :] + shift
        mxn = cfg[x][j, :] + shift

        ax[0].plot(x_data, mzn, c=colors[j],  linewidth=1)
        ax[1].plot(x_data, myn, c=colors[j],  linewidth=1)
        ax[2].plot(x_data, mxn, c=colors[j],  linewidth=1)

        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx]

        ax[0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
        ax[1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
        ax[2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

        # optional: faint baseline marking each slice's zero-line, for reference
        for a in ax:
            a.axhline(shift, color=colors[j], lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift), xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=15, color=colors[j])
            
    ax[0].set_ylabel(r'$M_{z}$ (offset per slice)')
    ax[1].set_ylabel(r'$M_{y}$ (offset per slice)')
    ax[2].set_ylabel(r'$M_{x}$ (offset per slice)')
    for a in ax:
        a.set_xlabel('x (pix)')
        a.set_yticks([])  # offset y-values aren't physically meaningful, so hide the axis numbers

    fig.suptitle(f"{cfg['sample']} {cfg['coord']}-slice, colored by {select_psi} ψ")

    plt.tight_layout()
    plt.show()

for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        # plot_by_psi_bin_offset(crop_info['dw'], select_psi = 'my_', normalized = False, offset = 0.05)
        # plot_by_psi_bin_offset(crop_info['dw'], select_psi = 'mx_', normalized = False, offset = 0.05)
        plot_by_psi_bin_offset(crop_info['dw'], select_psi = '', normalized = False, offset = 0.05)

In [ ]:
# ### plot for full stack 
# for sample, coords in samples.items():
#     for coord, data_dict in coords.items():
#         #plot_by_psi_bin(samples[sample][coord], select_psi= 'my')
#         plot_by_psi_bin_offset(samples[sample][coord], select_psi= '', n_bins=11, offset=2.5)

In [ ]:
def plot_gradient_by_psi_bin_offset(cfg,select_psi = '', normalized = 'True', n_bins=11, offset=0.25):
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_over_pi = cfg[select_psi+'psi_grid'] / np.pi  # units of ψ/π, matching your existing convention

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(1, 3, figsize=(21, 8))

    n_slices = len(cfg['eta_targets'])
    if normalized: 
        x = 'dmxn_dx'
        y = 'dmyn_dx'
        z = 'dmzn_dx'
    else: 
        x = 'dmx_dx'
        y = 'dmy_dx'
        z = 'dmz_dx'


    for j, eta_v in enumerate(cfg['eta_targets']):
        psi_slice = psi_over_pi[j, :]

        # shift this slice's baseline up by j * offset
        shift = j * offset

        mzn = cfg[z][j, :] + shift
        myn = cfg[y][j, :] + shift
        mxn = cfg[x][j, :] + shift

        ax[0].plot(x_data, mzn, c=colors[j], linewidth=1)
        ax[1].plot(x_data, myn, c=colors[j], linewidth=1)
        ax[2].plot(x_data, mxn, c=colors[j], linewidth=1)

        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx]

        ax[0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
        ax[1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
        ax[2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

        # optional: faint baseline marking each slice's zero-line, for reference
        for a in ax:
            a.axhline(shift, color='gray', lw=0.5, alpha=0.3, zorder=0)
            a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift), xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=15, color=colors[j])
            
    ax[0].set_ylabel(r'd$M_{z}$/dx (offset per slice)')
    ax[1].set_ylabel(r'd$M_{y}$/dx (offset per slice)')
    ax[2].set_ylabel(r'd$M_{x}$/dx (offset per slice)')
    for a in ax:
        a.set_xlabel('x (pix)')
        a.set_yticks([])  # offset y-values aren't physically meaningful, so hide the axis numbers

    fig.suptitle(f"{cfg['sample']} {cfg['coord']}-slice, colored by {select_psi} ψ")

    plt.tight_layout()
    plt.show()

for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        plot_gradient_by_psi_bin_offset(crop_info['dw'], select_psi = '', normalized = False, offset = 0.005)


In [ ]:
def plot_by_psi_sample_comparison(crop, coord, sample_ids, normalized = True, n_bins=11, offset=0.5, linestyles=None):
    if linestyles is None:
        linestyles = ['-', '--', ':', '-.']

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(2, 3, figsize=(21, 16))

    if normalized: 
        x = 'mxn_grid'
        y = 'myn_grid'
        z = 'mzn_grid'
        dx = 'dmxn_dx'
        dy = 'dmyn_dx'
        dz = 'dmzn_dx'
    else: 
        x = 'mx_grid'
        y = 'my_grid'
        z = 'mz_grid'
        dx = 'dmx_dx'
        dy = 'dmy_dx'
        dz = 'dmz_dx'


    for s_idx, sample in enumerate(sample_ids):
        cfg = crop[sample][coord]['dw']
        x_data = cfg['unique_x'] - cfg['unique_x'].mean()
        psi_over_pi = cfg['psi_grid'] / np.pi
        linestyle = linestyles[s_idx % len(linestyles)]

        for j, eta_v in enumerate(cfg['eta_targets']):
            psi_slice = psi_over_pi[j, :]
            shift = j * offset

            mzn = cfg[z][j, :] + shift
            myn = cfg[y][j, :] + shift
            mxn = cfg[x][j, :] + shift


            dmzn = cfg[dz][j, :] + shift
            dmyn = cfg[dy][j, :] + shift
            dmxn = cfg[dx][j, :] + shift

            ax[0,0].plot(x_data, mzn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[0,1].plot(x_data, myn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[0,2].plot(x_data, mxn, c=colors[j], linestyle=linestyle, linewidth=1)

            ax[1,0].plot(x_data, dmzn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[1,1].plot(x_data, dmyn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[1,2].plot(x_data, dmxn, c=colors[j], linestyle=linestyle, linewidth=1)

            bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
            bin_idx = np.clip(bin_idx, 0, n_bins - 1)
            point_colors = bin_colors[bin_idx]

            ax[0,0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
            ax[0,1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
            ax[0,2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

            ax[1,0].scatter(x_data, dmzn, c=point_colors, s=20, zorder=3)
            ax[1,1].scatter(x_data, dmyn, c=point_colors, s=20, zorder=3)
            ax[1,2].scatter(x_data, dmxn, c=point_colors, s=20, zorder=3)

            # only draw/label each offset baseline once, using the first sample's eta_targets
            if s_idx == 0:
                for a in ax.flat:
                    a.axhline(shift, color='gray', lw=0.5, alpha=0.3, zorder=0)
                    a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift), xycoords=('axes fraction', 'data'),
                            xytext=(-8, 0), textcoords='offset points',
                            ha='right', va='center', fontsize=15, color=colors[j])

    ax[0,0].set_ylabel(r'$M_{z}$')
    ax[0,1].set_ylabel(r'$M_{y}$')
    ax[0,2].set_ylabel(r'$M_{x}$')

    ax[1,0].set_ylabel(r'd$M_{z}$/dx')
    ax[1,1].set_ylabel(r'd$M_{y}$/dx')
    ax[1,2].set_ylabel(r'd$M_{x}$/dx')


    # legend mapping linestyle -> sample
    sample_handles = [
        plt.Line2D([0], [0], color='black', linestyle=linestyles[i % len(linestyles)], label=sample)
        for i, sample in enumerate(sample_ids)
    ]

    for a in ax.flat:
        a.set_xlabel('x (pix)')
        a.set_yticks([])
        a.legend(handles=sample_handles, fontsize=15, loc='upper right', title='sample')


   

    fig.suptitle(f'Sample comparison {coord}-slice')
    plt.tight_layout()
    plt.show()


for coord in ['x', 'y']:
    plot_by_psi_sample_comparison(crop1, coord, sample_ids, normalized = False)

In [ ]:
def plot_by_psi_sample_comparison_whighlight(crop, coord, sample_ids, normalized = True, n_bins=11, offset=0.05, psi_tol=0.05, linestyles=None):
    if linestyles is None:
        linestyles = ['-', '--', ':', '-.']

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    fig, ax = plt.subplots(2, 3, figsize=(21, 16))

    if normalized: 
        x = 'mxn_grid'
        y = 'myn_grid'
        z = 'mzn_grid'
        dx = 'dmxn_dx'
        dy = 'dmyn_dx'
        dz = 'dmzn_dx'
    else: 
        x = 'mx_grid'
        y = 'my_grid'
        z = 'mz_grid'
        dx = 'dmx_dx'
        dy = 'dmy_dx'
        dz = 'dmz_dx'


    for s_idx, sample in enumerate(sample_ids):
        cfg = crop[sample][coord]['dw']
        x_data = cfg['unique_x'] - cfg['unique_x'].mean()
        psi_over_pi = cfg['psi_grid'] / np.pi
        linestyle = linestyles[s_idx % len(linestyles)]


        n_slices = len(cfg['eta_targets'])
        neel_pct_per_eta = np.zeros(n_slices)
        bloch_pct_per_eta = np.zeros(n_slices)



        for j, eta_v in enumerate(cfg['eta_targets']):
            psi_slice = psi_over_pi[j, :]
            shift = j * offset

            mzn = cfg[z][j, :] + shift
            myn = cfg[y][j, :] + shift
            mxn = cfg[x][j, :] + shift


            dmzn = cfg[dz][j, :] + shift
            dmyn = cfg[dy][j, :] + shift
            dmxn = cfg[dx][j, :] + shift

            ax[0,0].plot(x_data, mzn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[0,1].plot(x_data, myn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[0,2].plot(x_data, mxn, c=colors[j], linestyle=linestyle, linewidth=1)

            ax[1,0].plot(x_data, dmzn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[1,1].plot(x_data, dmyn, c=colors[j], linestyle=linestyle, linewidth=1)
            ax[1,2].plot(x_data, dmxn, c=colors[j], linestyle=linestyle, linewidth=1)

            bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
            bin_idx = np.clip(bin_idx, 0, n_bins - 1)
            point_colors = bin_colors[bin_idx]

            # Néel-like: ψ≈0 OR ψ≈±1 (both point along ±x, sin(ψ)=0 either way)
            neel_mask = (np.abs(psi_slice) <= psi_tol) | (np.isclose(np.abs(psi_slice), 1, atol=psi_tol))
            # Bloch-like: ψ≈±0.5 (i.e. ±π/2)
            bloch_mask = np.isclose(np.abs(psi_slice), 0.5, atol=psi_tol)

            neel_pct_per_eta[j] = 100 * neel_mask.sum() / psi_slice.size #neel_mask.size
            bloch_pct_per_eta[j] = 100 * bloch_mask.sum() / psi_slice.size #bloch_mask.size

            point_colors[neel_mask] = mcolors.to_rgba('cyan')
            point_colors[bloch_mask] = mcolors.to_rgba('magenta')

            ax[0,0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
            ax[0,1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
            ax[0,2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

            ax[1,0].scatter(x_data, dmzn, c=point_colors, s=20, zorder=3)
            ax[1,1].scatter(x_data, dmyn, c=point_colors, s=20, zorder=3)
            ax[1,2].scatter(x_data, dmxn, c=point_colors, s=20, zorder=3)


            # only draw/label each offset baseline once, using the first sample's eta_targets
            if s_idx == 0:
                for a in ax.flat:
                    a.axhline(shift, color=colors[j], lw=1, alpha=0.3, zorder=0)
                    a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift), xycoords=('axes fraction', 'data'),
                            xytext=(-8, 0), textcoords='offset points',
                            ha='right', va='center', fontsize=15, color=colors[j])
                    

    ax[0,0].set_ylabel(r'$M_{z}$')
    ax[0,1].set_ylabel(r'$M_{y}$')
    ax[0,2].set_ylabel(r'$M_{x}$')

    ax[1,0].set_ylabel(r'd$M_{z}$/dx')
    ax[1,1].set_ylabel(r'd$M_{y}$/dx')
    ax[1,2].set_ylabel(r'd$M_{x}$/dx')


    # legend mapping linestyle -> sample
    sample_handles = [
        plt.Line2D([0], [0], color='black', linestyle=linestyles[i % len(linestyles)], label=sample)
        for i, sample in enumerate(sample_ids)
    ]

    for a in ax.flat:
        a.set_xlabel('x (pix)')
        a.set_yticks([])
        a.legend(handles=sample_handles, fontsize=15, loc='upper right', title='sample')


   

    fig.suptitle(f'Sample comparison {coord}-slice')
    plt.tight_layout()
    plt.show()


for coord in ['x', 'y']:
    plot_by_psi_sample_comparison_whighlight(crop1, coord, sample_ids, normalized = False)

In [ ]:

def plot_psi_bin_offset_w_highlight(cfg, select_psi='', normalized = False, n_bins=11, offset=1.5, psi_tol=0.05):
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    psi_key = select_psi + 'psi_grid'
    psi_over_pi = cfg[psi_key] / np.pi  # units of ψ/π

    bin_edges = np.linspace(-1, 1, n_bins + 1)
    cmap = plt.cm.twilight_shifted
    bin_colors = cmap(np.linspace(0, 1, n_bins))

    n_slices = len(cfg['eta_targets'])
    neel_pct_per_eta = np.zeros(n_slices)
    bloch_pct_per_eta = np.zeros(n_slices)


    if normalized: 
        x = 'mxn_grid'
        y = 'myn_grid'
        z = 'mzn_grid'
        dx = 'dmxn_dx'
        dy = 'dmyn_dx'
        dz = 'dmzn_dx'
    else: 
        x = 'mx_grid'
        y = 'my_grid'
        z = 'mz_grid'
        dx = 'dmx_dx'
        dy = 'dmy_dx'
        dz = 'dmz_dx'
    

    fig, ax = plt.subplots(1, 3, figsize=(26, 8))

    for j, eta_v in enumerate(cfg['eta_targets']):
        psi_slice = psi_over_pi[j, :]
        shift = j * offset

        mzn = cfg[z][j, :] + shift
        myn = cfg[y][j, :] + shift
        mxn = cfg[x][j, :] + shift

        ax[0].plot(x_data, mzn, c=colors[j], linewidth=1)
        ax[1].plot(x_data, myn, c=colors[j], linewidth=1)
        ax[2].plot(x_data, mxn, c=colors[j], linewidth=1)

        bin_idx = np.digitize(psi_slice, bin_edges[1:-1])
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)
        point_colors = bin_colors[bin_idx].copy()

        # Néel-like: ψ≈0 OR ψ≈±1 (both point along ±x, sin(ψ)=0 either way)
        neel_mask = (np.abs(psi_slice) <= psi_tol) | (np.isclose(np.abs(psi_slice), 1, atol=psi_tol)) | (np.isclose(np.abs(psi_slice), -1, atol=psi_tol))
        # Bloch-like: ψ≈±0.5 (i.e. ±π/2)
        bloch_mask = np.isclose(np.abs(psi_slice), 0.5, atol=psi_tol)

        neel_pct_per_eta[j] = 100 * neel_mask.sum() / psi_slice.size #neel_mask.size
        bloch_pct_per_eta[j] = 100 * bloch_mask.sum() / psi_slice.size #bloch_mask.size

        point_colors[neel_mask] = mcolors.to_rgba('cyan')
        point_colors[bloch_mask] = mcolors.to_rgba('magenta')

        ax[0].scatter(x_data, mzn, c=point_colors, s=20, zorder=3)
        ax[1].scatter(x_data, myn, c=point_colors, s=20, zorder=3)
        ax[2].scatter(x_data, mxn, c=point_colors, s=20, zorder=3)

        for a in ax:
            a.axhline(shift, color=colors[j], lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift), xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=15, color=colors[j])

    # overall percentages, computed across all slices combined
    neel_mask_all = (np.abs(psi_over_pi) <= psi_tol) | (np.isclose(np.abs(psi_over_pi), 1, atol=psi_tol))
    bloch_mask_all = np.isclose(np.abs(psi_over_pi), 0.5, atol=psi_tol)
    neel_pct_overall = 100 * neel_mask_all.sum() / psi_over_pi.size #neel_mask_all.size
    bloch_pct_overall = 100 * bloch_mask_all.sum() / psi_over_pi.size #bloch_mask_all.size

    # save everything to cfg for later cross-sample comparison
    cfg['neel_pct_overall'] = neel_pct_overall
    cfg['bloch_pct_overall'] = bloch_pct_overall
    cfg['neel_pct_per_eta'] = neel_pct_per_eta
    cfg['bloch_pct_per_eta'] = bloch_pct_per_eta

    ax[0].set_ylabel(r'$M_{z}$ (offset per slice)')
    ax[1].set_ylabel(r'$M_{y}$ (offset per slice)')
    ax[2].set_ylabel(r'$M_{x}$ (offset per slice)')
    for a in ax:
        a.set_xlabel('x (pix)')
        a.set_yticks([])

    # # legend: overall summary first, then per-eta_target breakdown
    # legend_handles = [
    #     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='cyan', markersize=10,
    #                label=fr'Néel-like (ψ≈0,±1) — overall: {neel_pct_overall:.1f}%'),
    #     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='magenta', markersize=10,
    #                label=fr'Bloch-like (ψ≈±0.5) — overall: {bloch_pct_overall:.1f}%'),
    # ]
    # for j, eta_v in enumerate(cfg['eta_targets']):
    #     legend_handles.append(
    #         plt.Line2D([0], [0], color=colors[j],
    #                    label=fr'{eta_v*100:.0f}%: Néel {neel_pct_per_eta[j]:.1f}% vs. Bloch {bloch_pct_per_eta[j]:.1f}%')
    #     )

    # ax[2].legend(handles=legend_handles, fontsize=15, loc='upper left',
    #              bbox_to_anchor=(1.02, 1.0), borderaxespad=0, reversed(legend_handles))

    # fig.suptitle(f"{cfg['sample']} {cfg['coord']}-slice, colored by {select_psi} ψ")

    per_eta_handles = [
        plt.Line2D([0], [0], color=colors[j],
                label=fr'{eta_v*100:.0f}%: Néel {neel_pct_per_eta[j]:.1f}% vs. Bloch {bloch_pct_per_eta[j]:.1f}%')
        for j, eta_v in enumerate(cfg['eta_targets'])
    ]

    legend_handles = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='cyan', markersize=10,
                label=fr'Néel-like (ψ≈0,±1) — overall: {neel_pct_overall:.1f}%'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='magenta', markersize=10,
                label=fr'Bloch-like (ψ≈±0.5) — overall: {bloch_pct_overall:.1f}%'),
    ] + list(reversed(per_eta_handles))

    ax[2].legend(handles=legend_handles, fontsize=15, loc='upper left',
                bbox_to_anchor=(1.02, 1.0), borderaxespad=0)
    
    fig.suptitle(f"{cfg['sample']} {cfg['coord']}-slice, colored by {select_psi} ψ")

    plt.tight_layout()
    plt.show()
    return cfg




for sample, coords in samples.items():
    for coord, data_dict in coords.items():
        plot_psi_bin_offset_w_highlight(samples[sample][coord], select_psi= '', normalized = False, n_bins=11, offset=2.5)

In [ ]:


# def plot_neel_bloch_comparison(samples, sample_ids, offset=15):

#     fig, ax = plt.subplots(1, 3, figsize=(21, 7))

#     overall_neel = []
#     overall_bloch = []

#     for s_idx, sample in enumerate(sample_ids):
#         shift = s_idx * offset
#         print(shift)

#         for coord, a in zip(['y', 'x'], ax[:2]):
#             cfg = samples[sample][coord]
#             eta_pct = cfg['eta_targets'] * 100

#             a.plot(eta_pct, cfg['neel_pct_per_eta']+shift, color='cyan', marker='o', lw=1.5)
#             a.plot(eta_pct, cfg['bloch_pct_per_eta']+shift, color='magenta', marker='o', lw=1.5)

#         for a in ax[:2]:

#             avg_plot = (cfg['neel_pct_per_eta'][0] + shift + cfg['bloch_pct_per_eta'][0] + shift) / 2
#             a.axhline(avg_plot, color='k', lw=1, alpha=0.3, zorder=0)
#             a.annotate(fr'{sample}', xy=(0.15, avg_plot+1), xycoords=('axes fraction', 'data'),
#                     xytext=(-8, 0), textcoords='offset points',
#                     ha='right', va='center', fontsize=15, color='k')

#         # grab overall percentages once per sample (using 'y' coord as the reference;
#         # adjust if you want 'x', or want both shown separately)
#         overall_neel.append(samples[sample]['y']['neel_pct_overall'])
#         overall_bloch.append(samples[sample]['y']['bloch_pct_overall'])

#     ax[0].set_title('y-slice')
#     # ax[0].set_ylim(5, 20)
#     ax[1].set_title('x-slice')
#     # ax[1].set_ylim(5, 20)
#     for a in ax[:2]:
#         a.set_xlabel('Fraction of local thickness (%)')
#         a.set_ylabel('DW type (%)')

#     # sample_handles = [
#     #     plt.Line2D([0], [0], color='black', linestyle=linestyles[i % len(linestyles)], label=sample)
#     #     for i, sample in enumerate(sample_ids)
#     # ]
#     color_handles = [
#         plt.Line2D([0], [0], color='cyan', lw=2, label='% Néel'),
#         plt.Line2D([0], [0], color='magenta', lw=2, label='% Bloch'),
#     ]
#     # ax[0].legend(handles=color_handles , fontsize=8, loc='upper left')
#     # ax[1].legend(handles=color_handles , fontsize=8, loc='upper left')

#     x_pos = np.arange(len(sample_ids))
#     width = 0.35
#     ax[2].bar(x_pos - width/2, overall_neel, width, color='cyan', alpha = 0.6, label='% Néel')
#     ax[2].bar(x_pos + width/2, overall_bloch, width, color='magenta', alpha = 0.6, label='% Bloch')
#     ax[2].set_xticks(x_pos)
#     ax[2].set_xticklabels(sample_ids)
#     ax[2].set_xlabel('sample')
#     ax[2].set_ylabel('DW type (%)')
#     ax[2].set_title('Percent Néel/Bloch per sample')
#     ax[2].legend(fontsize=15)

#     plt.tight_layout()
#     plt.show()


def plot_neel_bloch_comparison(samples, sample_ids, offset=15):

    fig, ax = plt.subplots(1, 4, figsize=(30, 7))

    overall_neel, overall_bloch = [], []
    overall_neel_err, overall_bloch_err = [], []

    overall_neelx, overall_blochx = [], []
    overall_neel_errx, overall_bloch_errx = [], []

    for s_idx, sample in enumerate(sample_ids):
        shift = s_idx * offset

        for coord, a in zip(['y', 'x'], ax[:2]):
            cfg = samples[sample][coord]
            eta_pct = cfg['eta_targets'] * 100
            n_cols = cfg['n_cols']  # points per eta-level slice

            p_neel = cfg['neel_pct_per_eta'] / 100
            p_bloch = cfg['bloch_pct_per_eta'] / 100
            neel_err = np.sqrt(p_neel * (1 - p_neel) / n_cols) * 100
            bloch_err = np.sqrt(p_bloch * (1 - p_bloch) / n_cols) * 100

            a.errorbar(eta_pct, cfg['neel_pct_per_eta'] + shift, yerr=neel_err,
                       color='cyan', marker='o', lw=1.5, capsize=3)
            a.errorbar(eta_pct, cfg['bloch_pct_per_eta'] + shift, yerr=bloch_err,
                       color='magenta', marker='o', lw=1.5, capsize=3)

        for a in ax[:2]:
            avg_plot = (cfg['neel_pct_per_eta'][0] + shift + cfg['bloch_pct_per_eta'][0] + shift) / 2
            a.axhline(avg_plot, color='k', lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'{sample}', xy=(0.15, avg_plot + 2), xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=15, color='k')

        # overall percentages + their error bars, using 'y' coord as the reference
        cfg_y = samples[sample]['y']
        n_total = cfg_y['n_cols'] * len(cfg_y['eta_targets'])

        p_neel_all = cfg_y['neel_pct_overall'] / 100
        p_bloch_all = cfg_y['bloch_pct_overall'] / 100

        overall_neel.append(cfg_y['neel_pct_overall'])
        overall_bloch.append(cfg_y['bloch_pct_overall'])
        overall_neel_err.append(np.sqrt(p_neel_all * (1 - p_neel_all) / n_total) * 100)
        overall_bloch_err.append(np.sqrt(p_bloch_all * (1 - p_bloch_all) / n_total) * 100)



        # overall percentages + their error bars, using 'y' coord as the reference
        cfg_x = samples[sample]['x']
        n_totalx = cfg_x['n_cols'] * len(cfg_y['eta_targets'])

        p_neel_allx = cfg_x['neel_pct_overall'] / 100
        p_bloch_allx = cfg_x['bloch_pct_overall'] / 100

        overall_neelx.append(cfg_x['neel_pct_overall'])
        overall_blochx.append(cfg_x['bloch_pct_overall'])
        overall_neel_errx.append(np.sqrt(p_neel_allx * (1 - p_neel_allx) / n_totalx) * 100)
        overall_bloch_errx.append(np.sqrt(p_bloch_allx * (1 - p_bloch_allx) / n_totalx) * 100)




    ax[0].set_title('y-slice')
    ax[1].set_title('x-slice')
    for a in ax[:2]:
        a.set_xlabel('Fraction of local thickness (%)')
        a.set_ylabel('DW type (%)')

    color_handles = [
        plt.Line2D([0], [0], color='cyan', lw=2, label='% Néel'),
        plt.Line2D([0], [0], color='magenta', lw=2, label='% Bloch'),
    ]
    ax[0].legend(handles=color_handles, fontsize=8, loc='upper left')
    ax[1].legend(handles=color_handles, fontsize=8, loc='upper left')

    x_pos = np.arange(len(sample_ids))
    width = 0.35
    ax[2].bar(x_pos - width/2, overall_neel, width, yerr=overall_neel_err, capsize=4,
              color='cyan', alpha=0.6, label='% Néel')
    ax[2].bar(x_pos + width/2, overall_bloch, width, yerr=overall_bloch_err, capsize=4,
              color='magenta', alpha=0.6, label='% Bloch')
    ax[2].set_xticks(x_pos)
    ax[2].set_xticklabels(sample_ids)
    ax[2].set_xlabel('sample')
    ax[2].set_ylabel('DW type (%)')
    ax[2].set_title('Percent Néel/Bloch per sample \n y-slice')
    ax[2].legend(fontsize=8)




    ax[3].bar(x_pos - width/2, overall_neelx, width, yerr=overall_neel_errx, capsize=4,
              color='cyan', alpha=0.6, label='% Néel')
    ax[3].bar(x_pos + width/2, overall_blochx, width, yerr=overall_bloch_errx, capsize=4,
              color='magenta', alpha=0.6, label='% Bloch')
    ax[3].set_xticks(x_pos)
    ax[3].set_xticklabels(sample_ids)
    ax[3].set_xlabel('sample')
    ax[3].set_ylabel('DW type (%)')
    ax[3].set_title('Percent Néel/Bloch per sample\n x-slice')
    ax[3].legend(fontsize=8)

    plt.tight_layout()
    plt.show()



plot_neel_bloch_comparison(samples, sample_ids)

In [ ]:
def plot_neel_bloch_comparison(samples, sample_ids, offset=15):

    fig, ax = plt.subplots(2, 2, figsize=(16, 14))

    overall_neel, overall_bloch = [], []
    overall_neel_err, overall_bloch_err = [], []

    overall_neelx, overall_blochx = [], []
    overall_neel_errx, overall_bloch_errx = [], []

    for s_idx, sample in enumerate(sample_ids):
        shift = s_idx * offset

        # per-eta line plots: y-slice -> ax[0,0], x-slice -> ax[1,0]
        for coord, a in zip(['y', 'x'], [ax[0, 0], ax[1, 0]]):
            cfg = samples[sample][coord]
            eta_pct = cfg['eta_targets'] * 100
            n_cols = cfg['n_cols']

            p_neel = cfg['neel_pct_per_eta'] / 100
            p_bloch = cfg['bloch_pct_per_eta'] / 100
            neel_err = np.sqrt(p_neel * (1 - p_neel) / n_cols) * 100
            bloch_err = np.sqrt(p_bloch * (1 - p_bloch) / n_cols) * 100

            a.errorbar(eta_pct, cfg['neel_pct_per_eta'] + shift, yerr=neel_err,
                       color='cyan', marker='o', lw=1.5, capsize=3)
            a.errorbar(eta_pct, cfg['bloch_pct_per_eta'] + shift, yerr=bloch_err,
                       color='magenta', marker='o', lw=1.5, capsize=3)

        # baseline + sample label, on both per-eta subplots
        for coord, a in zip(['y', 'x'], [ax[0, 0], ax[1, 0]]):
            cfg = samples[sample][coord]
            avg_plot = (cfg['neel_pct_per_eta'][0] + shift + cfg['bloch_pct_per_eta'][0] + shift) / 2
            a.axhline(avg_plot, color='k', lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'{sample}', xy=(0.15, avg_plot + 2), xycoords=('axes fraction', 'data'),
                       xytext=(-8, 0), textcoords='offset points',
                       ha='right', va='center', fontsize=15, color='k')

        # overall percentages + error bars — y-slice
        cfg_y = samples[sample]['y']
        n_total = cfg_y['n_cols'] * len(cfg_y['eta_targets'])
        p_neel_all = cfg_y['neel_pct_overall'] / 100
        p_bloch_all = cfg_y['bloch_pct_overall'] / 100

        overall_neel.append(cfg_y['neel_pct_overall'])
        overall_bloch.append(cfg_y['bloch_pct_overall'])
        overall_neel_err.append(np.sqrt(p_neel_all * (1 - p_neel_all) / n_total) * 100)
        overall_bloch_err.append(np.sqrt(p_bloch_all * (1 - p_bloch_all) / n_total) * 100)

        # overall percentages + error bars — x-slice
        cfg_x = samples[sample]['x']
        n_totalx = cfg_x['n_cols'] * len(cfg_x['eta_targets'])
        p_neel_allx = cfg_x['neel_pct_overall'] / 100
        p_bloch_allx = cfg_x['bloch_pct_overall'] / 100

        overall_neelx.append(cfg_x['neel_pct_overall'])
        overall_blochx.append(cfg_x['bloch_pct_overall'])
        overall_neel_errx.append(np.sqrt(p_neel_allx * (1 - p_neel_allx) / n_totalx) * 100)
        overall_bloch_errx.append(np.sqrt(p_bloch_allx * (1 - p_bloch_allx) / n_totalx) * 100)

    # --- top row: y-slice ---
    ax[0, 0].set_title('y-slice ')
    ax[0, 0].set_xlabel('Fraction of local thickness (%)')
    ax[0, 0].set_ylabel('DW type (%)')

    color_handles = [
        plt.Line2D([0], [0], color='cyan', lw=2, label='% Néel'),
        plt.Line2D([0], [0], color='magenta', lw=2, label='% Bloch'),
    ]
    ax[0, 0].legend(handles=color_handles, fontsize=8, loc='upper left')

    x_pos = np.arange(len(sample_ids))
    width = 0.35
    ax[0, 1].bar(x_pos - width/2, overall_neel, width, yerr=overall_neel_err, capsize=4,
                 color='cyan', alpha=0.6, label='% Néel')
    ax[0, 1].bar(x_pos + width/2, overall_bloch, width, yerr=overall_bloch_err, capsize=4,
                 color='magenta', alpha=0.6, label='% Bloch')
    ax[0, 1].set_xticks(x_pos)
    ax[0, 1].set_xticklabels(sample_ids)
    ax[0, 1].set_xlabel('sample')
    ax[0, 1].set_ylabel('DW type (%)')
    ax[0, 1].set_title('Overall Néel/Bloch % per sample\ny-slice')
    ax[0, 1].legend(fontsize=8)

    # --- bottom row: x-slice ---
    ax[1, 0].set_title('x-slice ')
    ax[1, 0].set_xlabel('Fraction of local thickness (%)')
    ax[1, 0].set_ylabel('DW type (%)')
    ax[1, 0].legend(handles=color_handles, fontsize=8, loc='upper left')

    ax[1, 1].bar(x_pos - width/2, overall_neelx, width, yerr=overall_neel_errx, capsize=4,
                 color='cyan', alpha=0.6, label='% Néel')
    ax[1, 1].bar(x_pos + width/2, overall_blochx, width, yerr=overall_bloch_errx, capsize=4,
                 color='magenta', alpha=0.6, label='% Bloch')
    ax[1, 1].set_xticks(x_pos)
    ax[1, 1].set_xticklabels(sample_ids)
    ax[1, 1].set_xlabel('sample')
    ax[1, 1].set_ylabel('DW type (%)')
    ax[1, 1].set_title('Overall Néel/Bloch % per sample\nx-slice')
    ax[1, 1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()


plot_neel_bloch_comparison(samples, sample_ids)

# Calculate spatial energy variation like in Legrand

In [ ]:
samples['665']['y'].keys()

In [ ]:
# def calc_energies(cfg, coord, A=1.0, D=1.0, K=1.0):
#     """
#     Computes exchange, DMI, and PMA energy densities per x-position, per eta slice.
#     Assumes cfg already has 'mxn_grid', 'mzn_grid', 'unique_x' (from straighten_and_store).

#     Exchange: -A * (dmx/dx)^2         [divergence along x only, per your spec]
#     DMI:       D * [m(x) x m(x+dx)]_z   (discrete neighbor cross product, z-component)
#     PMA:      -K * mz^2               [z . m = mz, assuming z is your out-of-plane axis]
#     """

#     mx = cfg[coord]['mxn_grid']
#     my = cfg[coord]['myn_grid']
#     mz = cfg[coord]['mzn_grid']

#     print(cfg['sample'])
#     print(coord)

#     if coord == 'x': 
#         print('select x')
#         dm_dx = cfg['coord']['dmxn_dx']

#         my_shift = np.roll(my, -1, axis=1)
#         mz_shift = np.roll(mz, -1, axis=1)
#         cross = my * mz_shift - mz * my_shift


#     elif coord == 'y': 
#         print('select y')
#         dm_dx = cfg['coord']['dmyn_dx']

#         mx_shift = np.roll(mx, -1, axis=1)
#         mz_shift = np.roll(mz, -1, axis=1)
#         cross = mz * mx_shift - mx * mz_shift


#     exchange_grid = -A * (dm_dx ** 2)
#     dmi_grid = D * cross
#     dmi_grid[:, -1] = np.nan
#     pma_grid = -K * (mz ** 2)

#     cfg['exchange_grid'] = exchange_grid
#     cfg['dmi_grid'] = dmi_grid
#     cfg['pma_grid'] = pma_grid
#     cfg['A'], cfg['D'], cfg['K'] = A, D, K  # store constants used, for traceability

    # return cfg




def calc_energies(cfg, coord, A=1.0, D=1.0, K=1.0):
    """
    Computes exchange, DMI, and PMA energy densities per x-position, per eta slice.
    Assumes cfg already has 'mxn_grid', 'myn_grid', 'mzn_grid', 'unique_x'.

    Exchange: -A * (dm/dx)^2   [component depends on slice coord]
    DMI:       D * [m(x) x m(x+dx)]_component
    PMA:      -K * mz^2
    """
    mx = cfg['mxn_grid']
    my = cfg['myn_grid']
    mz = cfg['mzn_grid']


    if coord == 'x':
        # dm_dx = np.gradient(mx, dx, axis=1)
        dm_dx = cfg['dmxn_dx']

        my_shift = np.roll(my, -1, axis=1)
        mz_shift = np.roll(mz, -1, axis=1)
        cross = my * mz_shift - mz * my_shift

    elif coord == 'y':
        dm_dx = cfg['dmyn_dx']

        mx_shift = np.roll(mx, -1, axis=1)
        mz_shift = np.roll(mz, -1, axis=1)
        cross = mz * mx_shift - mx * mz_shift

    else:
        raise ValueError(f"Unrecognized coord '{coord}' — expected 'x' or 'y'")

    exchange_grid = -A * (dm_dx ** 2)
    dmi_grid = D * cross
    dmi_grid[:, -1] = np.nan  # last column wraps via np.roll — not physically valid

    pma_grid = -K * (mz ** 2)

    cfg['exchange_grid'] = exchange_grid
    cfg['dmi_grid'] = dmi_grid
    cfg['pma_grid'] = pma_grid
    cfg['A'], cfg['D'], cfg['K'] = A, D, K

    return cfg


# # full slices <--- need to calculate gradients here
# for sample, coords in samples.items():
#     for coord, cfg in coords.items():
#         samples[sample][coord] = calc_energies(cfg, coord, A=1.0, D=1.0, K=1.0)

for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        crop1[sample][coord] = calc_energies(crop_info, coord, A=1.0, D=1.0, K=1.0)

In [ ]:
def plot_energies(samples, sample, coord='y', offset=None, linestyles=None):
    if linestyles is None:
        linestyles = ['-', '--', ':', '-.']

    energy_keys = ['pma_grid', 'exchange_grid', 'dmi_grid']
    offset_shift = [1, 20, 4]
    # energy_labels = [r'PMA $-K m_z^2$',
    #                  r'Exchange $-A(\partial m_x/\partial x)^2$',
    #                  r'DMI $D(m_x \partial m_z/\partial x - m_z \partial m_x/\partial x)$']

    energy_labels = [r'PMA',
                     r'Exchange',
                     r'DMI']

    fig, ax = plt.subplots(1, 3, figsize=(21, 7))
    # fig, ax = plt.subplots(3, 1, figsize=(7, 21))


    cfg = samples[sample][coord]
    x_data = cfg['unique_x'] - cfg['unique_x'].mean()
    linestyle = '-'

    for j, eta_v in enumerate(cfg['eta_targets']):
        for a, key, ct in zip(ax, energy_keys, enumerate(offset_shift)):
            # print(ct)
            # print(offset_shift[ct[0]])

            shift = j * (offset if offset is not None else 0) / offset_shift[ct[0]]
            a.axhline(shift, color=colors[j], lw=1, alpha=0.3, zorder=0)
            a.annotate(fr'{eta_v*100:.0f}%', xy=(0.15, shift-(0.15/offset_shift[ct[0]])), xycoords=('axes fraction', 'data'),
                    xytext=(-8, 0), textcoords='offset points',
                    ha='right', va='center', fontsize=15, color=colors[j])

            values = cfg[key][j, :] + shift
            a.plot(x_data, values, color=colors[j], marker = 'o', linestyle=linestyle, lw=1.5)




    for a, label in zip(ax, energy_labels):
        a.set_xlabel('x (pix)')
        a.set_ylabel('Energy density (a.u.)')
        a.set_title(label)
        if offset is not None:
            a.set_yticks([])


    fig.suptitle(f'Calculated Energy Densties Across Slice \n {sample} {coord}-slice')
    plt.tight_layout()
    plt.show()


for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        plot_energies(crop1, sample, coord=coord, offset = 1)

In [ ]:
for sample, coords in crop1.items():
    for coord, crop_info in coords.items():
        print(sample, coord, type(crop_info), list(crop_info.keys()) if isinstance(crop_info, dict) else crop_info)

# Compare to basic fits

In [ ]:
def plot_profile_with_model(data_dict, wall_types, x0, delta_demo, field_pairs):
    field_to_comp = {'mz': 'Mz', 'my': 'My', 'mx': 'Mx'}
    comp_index = {'Mz': 2, 'My': 1, 'Mx': 0}  # adjust if magnetization_components returns (Mx, My, Mz)

    x_data = data_dict['unique_x']
    x_data = x_data - x_data.mean()

    for field, field_n in field_pairs:
        comp = field_to_comp[field]

        fig, ax1 = plt.subplots(1, 2, figsize=(15, 5))

        # --- experimental data ---
        for j, eta_v in enumerate(data_dict['eta_targets']):
            ax1[0].plot(x_data, data_dict[field_n + '_grid'][j, :],
                        color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')
            ax1[1].plot(x_data, data_dict['d' + field_n + '_dx'][j, :],
                        color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')

        # --- overlay theoretical wall model(s), computed fresh for this x0 ---
        for wall, wcfg in wall_types.items():
            psi = wcfg['psi']
            if np.isscalar(psi):
                Mx, My, Mz = vis.magnetization_components(x0, delta_demo, psi)
                values = {'Mx': Mx, 'My': My, 'Mz': Mz}[comp]
                dvalues = np.gradient(values, x0[1] - x0[0])

                ax1[0].plot(x0, values, color=wcfg['color'], lw=2,
                            label=f'{wall} model, ψ={psi:.2f} & Δ = {delta_demo*8:.2f} nm')
                ax1[1].plot(x0, dvalues, color=wcfg['color'], lw=2,
                            label=f'{wall} model, ψ={psi:.2f} & Δ = {delta_demo*8:.2f} nm')
            else:
                for psi_val, color in zip(psi, wcfg['colors']):
                    Mx, My, Mz = magnetization_components(x0, delta_demo, psi_val)
                    values = {'Mx': -Mx, 'My': My, 'Mz': Mz}[comp]  # keep your -Mx flip for hybrid
                    dvalues = np.gradient(values, x0[1] - x0[0])
                    label = rf'{wall} $\psi={np.degrees(psi_val):.0f}°$'
                    ax1[0].plot(x0, values, color=color, ls='--', lw=1, label=label)
                    ax1[1].plot(x0, dvalues, color=color, ls='--', lw=1, label=label)

        ax1[0].set_ylabel(r'$M$')
        ax1[0].set_xlabel('x (pix), centered')
        ax1[0].legend(fontsize=7, ncol=2, title='data / model', loc='upper right')
        ax1[0].set_title(f"{field} \n full {data_dict['coord']}-DW profile + model")

        ax1[1].set_xlabel('x (pix), centered')
        ax1[1].set_ylabel(r'$dM(x)/dx$')
        ax1[1].legend(fontsize=7, ncol=2, title='data / model', loc='upper right')
        ax1[1].set_title(f"{field} \n full {data_dict['coord']}-DW Gradient + model")

        plt.tight_layout()
        plt.show()
        
test = ['665', '270', '452']

# print(dw_fits['665'].keys())

for sample in test:
    xmax, xmin = crop1[sample]['xmax'], crop1[sample]['xmin']
    x0 = np.linspace(0, (xmax - xmin), (xmax - xmin) + 1)
    x0 = x0 - x0.mean()  # recenter around 0
    delta = np.mean(dw_fits[sample]['dw_width']) / 8 # pixel size


    
    plot_profile_with_model(crop1[sample]['dw'], wall_types, x0, delta, field_pairs)


# plot_profile_with_model(y_665_crop1_smooth, wall_types, x0, field_pairs)


# Update wall profile fit

In [ ]:
## update hybrid fit 
sigma = 10
pixel_size_nm = 8 
delta = np.array(dw_width) / pixel_size_nm
center = np.array(dw_center) / pixel_size_nm

psi_hybrid_list = [-np.pi / 9, -np.pi / 9, 0, np.pi / 4, np.pi / 4 ]

colors2 = plt.cm.viridis(np.linspace(0, 1, len(center))) #cool


def magnetization_components_w_center(x, delta, psi, center):
    """
    Compute Mx, My, Mz for a domain wall with:
      - wall normal along x
      - easy axis along z
      - azimuthal angle psi (psi=0: Neel, psi=pi/2: Bloch)
    Returns normalized components (M/Ms).
    """
    theta = theta_profile(x, delta)
    Mx = np.sin(theta) * np.cos(psi)
    My = np.sin(theta) * np.sin(psi)
    Mz = np.cos(theta) 
    return Mx, My, Mz



# ---------- MZ ----------
fig, ax1 = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(rf'Updated Hybrid Wall fit from Hyperbolic Tan Fit Parameters', fontsize=20, y=1.0)
for j, eta_v in enumerate(eta_targets):
    smooth_mzn = gaussian_filter(mzn_grid[j, :], sigma = set_sigma)
    dMz_dx   = np.gradient(smooth_mzn, dx0)

    ___, ___, Mz_h = magnetization_components_w_center(x0, delta[j], psi_hybrid_list[j], center[j])
    dMzh_dx   = np.gradient(Mz_h, dx0)

    ax1[0].plot(x0,smooth_mzn, color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')
    ax1[1].plot(x0, dMz_dx, color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')
    

    ax1[0].plot(x0 + center[j], Mz_h, color=colors2[j], linestyle = ':') 
    ax1[1].plot(x0 + center[j], dMzh_dx, color=colors2[j], linestyle = ':')

ax1[0].set_ylabel(r'$M_z / M_{norm}$')
ax1[0].set_xlabel('x (pix)')
ax1[0].legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc = 'upper right')
ax1[0].set_title('Smoothed Domain Wall Profile')

ax1[1].set_xlabel('x (pix)')
ax1[1].set_ylabel(r'$dM_z(x)$/dx')
ax1[1].legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc = 'upper right')
ax1[1].set_title('Smoothed Gradient')

plt.tight_layout()
plt.show()

# ---------- MY ----------
fig, ax1 = plt.subplots(1, 2, figsize=(15, 5))
for j, eta_v in enumerate(eta_targets):
    smooth_myn = gaussian_filter(myn_grid[j, :], sigma = set_sigma)
    dMy_dx   = np.gradient(smooth_myn, dx0)

    ___, My_h, __ = magnetization_components_w_center(x0, delta[j], psi_hybrid_list[j], center[j])
    dMyh_dx   = np.gradient(My_h, dx0)

    ax1[0].plot(x0, smooth_myn, color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')
    ax1[1].plot(x0, dMy_dx, color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')
    

    ax1[0].plot(x0 + center[j], My_h, color=colors2[j], linestyle = ':', label = f'ψ = {psi_hybrid_list[j]:.2f}') 
    ax1[1].plot(x0 + center[j], dMyh_dx, color=colors2[j], linestyle = ':')

ax1[0].set_ylabel(r'$M_y / M_{norm}$')
ax1[0].set_xlabel('x (pix)')
ax1[0].legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc = 'upper right')
ax1[0].set_title('Smoothed Domain Wall Profile')

ax1[1].set_xlabel('x (pix)')
ax1[1].set_ylabel(r'$dM_y(x)$/dx')
ax1[1].legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc = 'upper right')
ax1[1].set_title('Smoothed Gradient')

plt.tight_layout()
plt.show()

# ---------- MZ ----------
fig, ax1 = plt.subplots(1, 2, figsize=(15, 5))
for j, eta_v in enumerate(eta_targets):
    smooth_mxn = gaussian_filter(mxn_grid[j, :], sigma = set_sigma)
    dMx_dx   = np.gradient(smooth_mxn, dx0)

    Mx_h, ___, ___ = magnetization_components_w_center(x0, delta[j], psi_hybrid_list[j], center[j])
    if j == 0: 
        Mx_h = -Mx_h
    dMxh_dx   = np.gradient(Mx_h, dx0)

    ax1[0].plot(x0,smooth_mxn, color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')
    ax1[1].plot(x0, dMx_dx, color=colors[j], lw=1.5, label=fr'{eta_v*100:.0f}%')
    

    ax1[0].plot(x0 + center[j], Mx_h, color=colors2[j], linestyle = ':') 
    ax1[1].plot(x0 + center[j], dMxh_dx, color=colors2[j], linestyle = ':')

ax1[0].set_ylabel(r'$M_x / M_{norm}$')
ax1[0].set_xlabel('x (pix)')
ax1[0].legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc = 'upper right')
ax1[0].set_title('Smoothed Domain Wall Profile')

ax1[1].set_xlabel('x (pix)')
ax1[1].set_ylabel(r'$dM_x(x)$/dx')
ax1[1].legend(fontsize=8, ncol=2, title=r'fraction of local thickness', loc = 'upper right')
ax1[1].set_title('Smoothed Gradient')

plt.tight_layout()
plt.show()


